# 🎯 XGBoost Insulin Dosage Prediction

**Model:** XGBoost Regressor (GPU Accelerated)  
**Target:** `bolus` (Insulin dosage prediction)  
**Dataset:** ~10.6M rows, balanced (25% positive)

---

## 🔗 1. Google Drive'a Bağlan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive bağlandı!")

## 📦 2. Kütüphaneleri Kur ve İçe Aktar

In [ ]:
!pip install polars xgboost -q

In [ ]:
import polars as pl
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import time
import os

print(f"✅ Polars: {pl.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

In [ ]:
# GPU kontrolü - ZORUNLU!
import subprocess

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print("✅ GPU BULUNDU:")
    print(result.stdout.split('\n')[8])  # GPU adını göster
except:
    raise RuntimeError("❌ GPU BULUNAMADI! Runtime → Change runtime type → GPU seçin!")

## 📂 3. Konfigürasyon

In [ ]:
# ==============================================================================
# KONFİGÜRASYON
# ==============================================================================

BASE_PATH = "/content/drive/MyDrive/DiabetesKD5/data/processed"

# Dosyalar
INPUT_FILE = f"{BASE_PATH}/train_balanced_xgboost.parquet"
MODEL_OUTPUT = f"{BASE_PATH}/xgboost_insulin_model.json"

# Feature ve Target tanımları
FEATURES = ['CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend', 'cgm_delta_5min', 'cgm_delta_15min']
TARGET = 'bolus'

# Training parametreleri
TEST_SIZE = 0.20
RANDOM_SEED = 42
SCALE_POS_WEIGHT = 3.0  # 1:3 dengesizlik için

print("=" * 70)
print("🎯 XGBoost Insulin Dosage Prediction")
print("=" * 70)
print(f"\n📂 Input: {INPUT_FILE}")
print(f"📂 Model Output: {MODEL_OUTPUT}")
print(f"\n📊 Features: {FEATURES}")
print(f"🎯 Target: {TARGET}")

## 📖 4. Veriyi Yükle (Polars - Hızlı!)

In [ ]:
print("-" * 70)
print("📖 Veri yükleniyor (Polars)...")
print("-" * 70)

start_time = time.time()

# Polars ile hızlı yükleme
df = pl.read_parquet(INPUT_FILE)

load_time = time.time() - start_time

print(f"✅ Veri yüklendi: {len(df):,} satır × {len(df.columns)} sütun")
print(f"⏱️  Yükleme süresi: {load_time:.2f} saniye")
print(f"\n📊 Sütunlar: {df.columns}")

In [ ]:
# Veri özeti
n_positive = df.filter(pl.col(TARGET) > 0).height
n_negative = df.filter(pl.col(TARGET) == 0).height
balance_ratio = n_negative / n_positive if n_positive > 0 else 0

print(f"\n📊 Sınıf Dağılımı:")
print(f"   Pozitif (bolus > 0): {n_positive:,} ({n_positive/len(df)*100:.1f}%)")
print(f"   Negatif (bolus = 0): {n_negative:,} ({n_negative/len(df)*100:.1f}%)")
print(f"   Oran: 1:{balance_ratio:.1f}")

## 🔀 5. Train/Test Split

In [ ]:
print("-" * 70)
print("🔀 Train/Test split yapılıyor...")
print("-" * 70)

# Feature ve Target ayır
X = df.select(FEATURES).to_numpy()
y = df.select(TARGET).to_numpy().ravel()

print(f"📊 X shape: {X.shape}")
print(f"📊 y shape: {y.shape}")

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    shuffle=True, 
    random_state=RANDOM_SEED
)

print(f"\n✅ Train set: {X_train.shape[0]:,} satır")
print(f"✅ Test set:  {X_test.shape[0]:,} satır")

# Bellek temizle
del df, X, y
import gc
gc.collect()

## 🚀 6. XGBoost DMatrix Oluştur (GPU)

In [ ]:
print("-" * 70)
print("🚀 XGBoost DMatrix oluşturuluyor (GPU)...")
print("-" * 70)

# DMatrix oluştur - GPU için optimize
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURES)

print(f"✅ DMatrix oluşturuldu")
print(f"   Train: {dtrain.num_row():,} rows")
print(f"   Test:  {dtest.num_row():,} rows")

## ⚙️ 7. XGBoost Parametreleri

In [ ]:
# ==============================================================================
# XGBOOST PARAMETRELERİ - GPU ACCELERATED
# ==============================================================================

params = {
    # Objective
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    
    # GPU Acceleration - ZORUNLU!
    'device': 'cuda',
    'tree_method': 'hist',  # GPU hist for speed
    
    # Model complexity
    'max_depth': 8,
    'min_child_weight': 5,
    'gamma': 0.1,
    
    # Learning
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    
    # Regularization
    'reg_alpha': 0.1,   # L1
    'reg_lambda': 1.0,  # L2
    
    # İmbalance handling
    'scale_pos_weight': SCALE_POS_WEIGHT,
    
    # Reproducibility
    'seed': RANDOM_SEED,
}

NUM_ROUNDS = 500
EARLY_STOPPING = 50

print("⚙️ XGBoost Parametreleri:")
for k, v in params.items():
    print(f"   {k}: {v}")
print(f"\n📊 Rounds: {NUM_ROUNDS}, Early Stopping: {EARLY_STOPPING}")

## 🏋️ 8. Model Eğitimi

In [ ]:
print("=" * 70)
print("🏋️ XGBoost EĞİTİMİ BAŞLIYOR (GPU)")
print("=" * 70)

train_start = time.time()

# Eğitim
evals = [(dtrain, 'train'), (dtest, 'test')]

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=50  # Her 50 round'da bir göster
)

train_time = time.time() - train_start

print("\n" + "=" * 70)
print(f"✅ EĞİTİM TAMAMLANDI!")
print(f"⏱️  Eğitim süresi: {train_time:.2f} saniye ({train_time/60:.1f} dakika)")
print(f"🌳 Best iteration: {model.best_iteration}")
print(f"📊 Best score: {model.best_score:.6f}")

## 📊 9. Model Değerlendirmesi

In [ ]:
print("=" * 70)
print("📊 MODEL DEĞERLENDİRMESİ")
print("=" * 70)

# Tahminler
y_pred = model.predict(dtest)

# Negatif tahminleri sıfırla (insulin negatif olamaz)
y_pred = np.maximum(y_pred, 0)

print(f"\n📊 Tahmin istatistikleri:")
print(f"   Min: {y_pred.min():.4f}")
print(f"   Max: {y_pred.max():.4f}")
print(f"   Mean: {y_pred.mean():.4f}")
print(f"   Std: {y_pred.std():.4f}")

In [ ]:
# ==============================================================================
# METRIC 1: GLOBAL RMSE & MAE (Tüm veriler)
# ==============================================================================

global_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
global_mae = mean_absolute_error(y_test, y_pred)

print("-" * 70)
print("📈 METRIC 1: Global Performans (Tüm Test Seti)")
print("-" * 70)
print(f"   RMSE: {global_rmse:.4f}")
print(f"   MAE:  {global_mae:.4f}")
print(f"   Test samples: {len(y_test):,}")

In [ ]:
# ==============================================================================
# METRIC 2: POSITIVE-ONLY MAE (⭐ EN ÖNEMLİ METRİK!)
# ==============================================================================

# Sadece gerçek bolus > 0 olan satırları filtrele
positive_mask = y_test > 0
y_test_positive = y_test[positive_mask]
y_pred_positive = y_pred[positive_mask]

positive_mae = mean_absolute_error(y_test_positive, y_pred_positive)
positive_rmse = np.sqrt(mean_squared_error(y_test_positive, y_pred_positive))

print("-" * 70)
print("⭐ METRIC 2: Positive-Only Performans (Bolus > 0)")
print("-" * 70)
print(f"   RMSE: {positive_rmse:.4f}")
print(f"   MAE:  {positive_mae:.4f}  ← EN ÖNEMLİ!")
print(f"   Positive samples: {len(y_test_positive):,}")
print(f"   Ortalama gerçek bolus: {y_test_positive.mean():.4f}")
print(f"   Ortalama tahmin: {y_pred_positive.mean():.4f}")

In [ ]:
# ==============================================================================
# ÖZET RAPOR
# ==============================================================================

print("\n" + "=" * 70)
print("📋 ÖZET PERFORMANS RAPORU")
print("=" * 70)

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│  GLOBAL METRİKLER (Tüm Test Seti: {len(y_test):,} satır)                  │
├─────────────────────────────────────────────────────────────────────┤
│  RMSE:                              {global_rmse:>12.4f}                   │
│  MAE:                               {global_mae:>12.4f}                   │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│  ⭐ POZİTİF-ONLY METRİKLER (bolus > 0: {len(y_test_positive):,} satır)            │
├─────────────────────────────────────────────────────────────────────┤
│  RMSE:                              {positive_rmse:>12.4f}                   │
│  MAE:                               {positive_mae:>12.4f}  ← KEY METRIC    │
└─────────────────────────────────────────────────────────────────────┘

🏋️ Eğitim Süresi: {train_time:.1f} saniye
🌳 Best Iteration: {model.best_iteration}
""")

## 📊 10. Feature Importance

In [ ]:
print("-" * 70)
print("📊 Feature Importance")
print("-" * 70)

importance = model.get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)

print("\n🏆 Feature Ranking (Gain):")
for i, (feat, score) in enumerate(importance_sorted, 1):
    bar = '█' * int(score / max(importance.values()) * 30)
    print(f"   {i}. {feat:20s} {score:>10.2f} {bar}")

## 💾 11. Modeli Kaydet

In [ ]:
print("-" * 70)
print("💾 Model kaydediliyor...")
print("-" * 70)

# JSON formatında kaydet
model.save_model(MODEL_OUTPUT)

# Dosya boyutunu kontrol et
model_size_mb = os.path.getsize(MODEL_OUTPUT) / (1024**2)

print(f"✅ Model kaydedildi: {MODEL_OUTPUT}")
print(f"📦 Dosya boyutu: {model_size_mb:.2f} MB")

## ✅ 12. Tamamlandı!

In [ ]:
print("\n" + "=" * 70)
print("🎉 XGBoost EĞİTİMİ BAŞARIYLA TAMAMLANDI!")
print("=" * 70)

print(f"""
📂 Kaydedilen dosya: {MODEL_OUTPUT}

📊 Sonuçlar:
   • Global MAE: {global_mae:.4f}
   • Positive-Only MAE: {positive_mae:.4f}
   
🚀 Modeli yüklemek için:

```python
import xgboost as xgb

model = xgb.Booster()
model.load_model("{MODEL_OUTPUT}")

# Tahmin
dmatrix = xgb.DMatrix(features)
predictions = model.predict(dmatrix)
```
""")

---

## 🔬 Bonus: Örnek Tahminler

In [ ]:
# Pozitif örneklerden rastgele 10 tanesini göster
print("\n🔬 Örnek Tahminler (Pozitif Vakalar):")
print("-" * 50)

np.random.seed(42)
sample_indices = np.random.choice(np.where(positive_mask)[0], size=min(10, sum(positive_mask)), replace=False)

print(f"{'Index':>8} | {'Gerçek':>10} | {'Tahmin':>10} | {'Hata':>10}")
print("-" * 50)

for idx in sample_indices:
    actual = y_test[idx]
    pred = y_pred[idx]
    error = abs(actual - pred)
    print(f"{idx:>8} | {actual:>10.4f} | {pred:>10.4f} | {error:>10.4f}")